In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import OpenAI
from langchain_core.output_parsers import ListOutputParser, JsonOutputParser
import json

In [2]:
llm=OpenAI()

In [3]:
prompt=PromptTemplate(
    template="list 3 countries in {continent} and their capitals",
    input_variables=["continent"]
)
print(llm.invoke(input=prompt.format(continent="asia")))



1. Japan - Tokyo
2. India - New Delhi
3. Thailand - Bangkok


In [5]:
output_parser=JsonOutputParser()
format_instructions=output_parser.get_format_instructions()
print(format_instructions)

Return a JSON object.


In [6]:
prompt=PromptTemplate(
    template="List 3 countries in {continent} and their capitals.\n {format_instructions}",
    input_variables=["continent"],
    partial_variables={"format_instructions":format_instructions}
)

In [7]:
print(prompt.format(continent="north america"))

List 3 countries in north america and their capitals.
 Return a JSON object.


In [8]:
output=llm.invoke(input=prompt.format(continent="America"))
print(output)



{
  "United States": "Washington D.C.",
  "Canada": "Ottawa",
  "Mexico": "Mexico City"
}


In [12]:
countries=output_parser.parse(output)
countries=json.dumps(countries)
print(type(countries))

<class 'str'>


#Pydantic output parser


In [15]:
from typing import List
from langchain_openai import OpenAI
from pydantic import BaseModel, Field, field_validator
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

In [13]:
model=OpenAI()

In [16]:
class Ticket(BaseModel):
    date: str= Field(description="show date")
    time: str= Field(description="show time")
    theater: str= Field(description="theater name")
    count: int= Field(description="number of tickets")
    movie: str= Field(description="preferred movie")

In [29]:
p_parser=PydanticOutputParser(pydantic_object=Ticket)

In [17]:
ticket_template='''
Book us a movie ticket for two this friday at 6:00pm,
choose any theater, it doesnt matter , sned the confirmation email,
our preferred movie is :{query}
format instructions:{format_instructions}'''

In [28]:
ticket_Prompt=PromptTemplate(
    template=ticket_template,
    input_variables=['query'],
    partial_variables={'format_instructions':p_parser.get_format_instructions()}
)

In [24]:
input=ticket_Prompt.format(query="durandhar")
print(input)


Book us a movie ticket for two this friday at 6:00pm,
choose any theater, it doesnt matter , sned the confirmation email,
our preferred movie is :durandhar
format instructions:The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"date": {"description": "show date", "title": "Date", "type": "string"}, "time": {"description": "show time", "title": "Time", "type": "string"}, "theater": {"description": "theater name", "title": "Theater", "type": "string"}, "count": {"description": "number of tickets", "title": "Count", "type": "integer"}, "movie": {"description": "preferred movie", 

In [25]:
output=model.invoke(str(input))

In [26]:
output

'\n{"date": "Friday", "time": "6:00pm", "theater": "Any", "count": 2, "movie": "Durandhar"}'

In [31]:
reservation=p_parser.parse(output)
print(reservation)

date='Friday' time='6:00pm' theater='Any' count=2 movie='Durandhar'


In [32]:
reservation

Ticket(date='Friday', time='6:00pm', theater='Any', count=2, movie='Durandhar')

In [ ]:
 #we have 3 main sections in langchain which are Langchain_core 
#tips and tricks of langchain verbose and callbacks